##Importing Functions

In [0]:
from pyspark.sql.functions import col, regexp_replace
from pyspark.sql.functions import upper,lower
from pyspark.sql.functions import length,trim
from pyspark.sql.functions import min  , max
print(' functions imported succesfully')

##reading the table

In [0]:
df=spark.read.table('workspace.bronze.pizzas')
print('table read successfully!')

##Bronze Layer Validation

In [0]:
print('check schema...')
df.printSchema()
print('check data types...')
print(df.dtypes)
print('check data sample...')
df.show(3)

##column operations

In [0]:
#naming convention is snake_case no need to change
#no bad data types but change as defensive approach
df=df.withColumn('pizza_id',col("pizza_id").cast('string')).withColumn('pizza_type_id',col("pizza_type_id").cast('string')).withColumn('size',col("size").cast('string')).withColumn('price',col("price").cast('double'))
print('data types changed succesfully!')

#handling trailling spaces
def trim_spaces(dataframe):
    for colname,coltype in dataframe.dtypes:
        if coltype=='string':
            count=dataframe.filter(length(trim(col(colname)))!=length(col(colname))).count()
            if count>0: 
                print(f'column {colname} contains {count} trailling spaces')
                dataframe=dataframe.withColumn(colname,trim(col(colname)))
                print(f'{colname} trimmed successfully!')
            else :
                print('checked successfully!')
                print(f'no trailling spaces found in {colname}')
            
    return dataframe
    
df=trim_spaces(df)

#check size column
#df.select('size').distinct().show()
df=df.withColumn('size',upper(col("size")))
print('size column updated to uppercase succesfully!')
df.select('size').distinct().collect()

#check max,min price 
df.select(max(col("price")), min(col("price"))).show()


##row operations

In [0]:
#duplicates check
df_unique_count=df.distinct().count()
df_count=df.count()
if df_unique_count==df_count:
    print('no duplicates found!')
    print('checked successfully!')
else:
    print('duplicates found!')
    df=df.distinct()
    print('duplicates handled successfully!')

print('-'*40)
#nulls check
df_anynulls_count=df.na.drop('any').count()
df_allnulls_count=df.na.drop('all').count()
if df_allnulls_count !=df_count or df_anynulls_count != df_count:
    print('nulls found!')
    print('rows count after removing rows containing ANY nulls',df_anynulls_count)
    print('rows count after removing rows containing ALL nulls',df_allnulls_count)
    raise ValueError('Null values detected in dataset. Pipeline halted.')
else:
    print('no nulls found!')
    print('checked successfully!')


##validation checks

In [0]:
print('-'*40)
print('Data validation checks')
print('-'*40)
validation_results=[]
def validate(colname,rule,condition) :
    invalid= df.filter(condition).count()
    validation_results.append({
        'column' : colname,
        'rule' : rule,
        'invalid_count' : invalid,
        'invalid_pct': round(invalid / df.count() * 100, 2)
    })

validate("price", "is negative", col("price") < 0)
spark.createDataFrame(validation_results).show(truncate=False)

##saving 

In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('workspace.silver.pizzas')
print('saved successfully!')

In [0]:
%sql
--sanity check
SELECT * FROM workspace.silver.pizzas LIMIT 10